
# Wossist — Customer Business Profiling

## Project Overview

This notebook is the second analytical component of the **Wossist Customer Segmentation** project.

While the previous notebook focused on RFM segmentation of purchasing customers, this analysis extends the perspective to the **entire customer base**, including customers who have not purchased an insurance policy.

The objective is to develop a customer-level dataset combining RFM segmentation with complementary business profiles, providing a broader understanding of customer characteristics and interactions with Wossist.

### Business Questions

The analysis addresses four main questions:

1. **Product Profile:** Which insurance products have customers purchased, and how diversified is their purchasing history?
2. **Channel Profile:** Through which sales channels do customers purchase insurance policies?
3. **Lifecycle Profile:** How far have customers progressed through the commercial journey, from registration to repeat purchasing?
4. **Engagement Profile:** How frequently do customers interact with Wossist, and through which channels?

### Analytical Output

The notebook produces a consolidated customer-level dataset by integrating:

- Customer attributes and identifiers.
- Product, Channel, Lifecycle, and Engagement profiles.
- RFM metrics, scores, and segments from the previous notebook.

The dataset includes both purchasing and non-purchasing customers. RFM attributes remain unavailable for customers without purchases.

**Final output:** `output/customer_business_profiling.csv.gz`

This dataset will support the final Power BI Customer Segmentation report, enabling business analysis across customer segments, products, channels, and lifecycle stages.



## Data Sources & Methodology

### Data Sources

Customer Business Profiling combines information from the Wossist synthetic insurance dataset:

| Dataset | Purpose |
|---|---|
| `customers` | Customer master data and registration information |
| `policies` | Purchased policies, products, sales channels, and premiums |
| `quotes` | Quotation activity and pre-purchase behavior |
| `interactions` | Recorded customer interactions and engagement |
| `customer_rfm` | RFM metrics and segments produced by the previous notebook |

### Analytical Population

The analysis uses **all registered customers** as its reference population.

Each customer is represented by a single record in the final output, regardless of whether they have purchased a policy, requested a quote, or recorded an interaction.

### Profiling Methodology

The four business profiles are derived independently from their respective data sources and subsequently integrated at the customer level.

Profiles describe observed customer behavior and should not be interpreted as predictions of future purchases, profitability, or churn.

Where relevant, the analysis distinguishes between the absence of recorded activity and missing information.



## Data Loading & Preparation

This section loads the source datasets required for Customer Business Profiling, including the RFM output generated in the previous notebook.

The analysis uses the complete registered customer population as its reference, integrating purchasing, quotation, and interaction data at the customer level.

Before developing the business profiles, dataset dimensions, available attributes, and customer identifiers are reviewed to ensure consistent integration.


In [17]:

import pandas as pd
import numpy as np

# GitHub repository base URL
REPO_URL = (
    "https://raw.githubusercontent.com/AntonioPennacchia/"
    "customer-analytics/main/insurance-customer-analytics/"
)

DATA_URL = REPO_URL + "dataset/"
RFM_URL = (
    REPO_URL
    + "analytics-projects/02-customer-segmentation/output/customer_rfm.csv.gz"
)

# Load source datasets
customers = pd.read_csv(DATA_URL + "customers.csv.gz")
policies = pd.read_csv(DATA_URL + "policies.csv.gz")
quotes = pd.read_csv(DATA_URL + "quotes.csv.gz")
interactions = pd.read_csv(DATA_URL + "customer_interactions.csv.gz")

# Load business dimensions
products = pd.read_csv(DATA_URL + "products.csv.gz")
sales_channels = pd.read_csv(DATA_URL + "sales_channels.csv.gz")

# Load RFM output
rfm = pd.read_csv(RFM_URL)

# Dataset overview
datasets = {
    "Customers": customers,
    "Policies": policies,
    "Quotes": quotes,
    "Interactions": interactions,
    "Products": products,
    "Sales Channels": sales_channels,
    "RFM": rfm
}

dataset_summary = pd.DataFrame([
    {
        "Dataset": name,
        "Rows": len(df),
        "Columns": df.shape[1]
    }
    for name, df in datasets.items()
])

display(
    dataset_summary.style.format({
        "Rows": "{:,.0f}"
    })
)


C:\Users\pennacchia\AppData\Local\Temp\ipykernel_5728\1292401422.py:19: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  quotes = pd.read_csv(DATA_URL + "quotes.csv.gz")


,Dataset,Rows,Columns
0,Customers,"631,079",10
1,Policies,"223,292",16
2,Quotes,"763,917",7
3,Interactions,"640,430",7
4,Products,6,4
5,Sales Channels,4,2
6,RFM,"198,073",9



## Product Profiling

### Methodological Approach

Product Profiling describes the insurance business lines purchased by each customer, independently of their RFM segment.

Wossist operates across three business lines — **Travel, Pet, and Home** — each offering two product variants.

The analysis uses purchased policies, rather than quotations, to identify observed purchasing behavior.

Customers are classified into three mutually exclusive profiles:

| Product Profile | Definition |
|---|---|
| **No Purchase** | No purchased policies |
| **Single Product** | Policies purchased within one business line |
| **Multi Product** | Policies purchased across two or more business lines |

Product diversification is measured at the **business-line level**, rather than by the number of policies or product variants.

For example, a customer purchasing both Travel Single Trip and Travel Annual remains a Single Product customer, whereas a customer purchasing Travel and Pet policies is classified as Multi Product.

### Product-Level Attributes

In addition to the overall Product Profile, the dataset retains the number of purchased policies for each of Wossist's six product variants.

These attributes are stored as separate numerical columns to facilitate filtering, customer selection, and cross-analysis in the final Power BI Customer Segmentation report.

The product-level attributes complement the broader business-line classification without introducing additional customer segments.

All product profiles describe observed purchases and do not imply future product preferences or purchasing intentions.


In [18]:

# Associate purchased policies with product information
policy_products = policies[
    ["customer_id", "policy_id", "product_id"]
].merge(
    products[
        ["product_id", "business_line", "product_type"]
    ],
    on="product_id",
    how="left",
    validate="many_to_one"
)

# Validate product mapping
assert policy_products["business_line"].notna().all()

# Count distinct purchased business lines per customer
product_summary = (
    policy_products.groupby("customer_id")
    .agg(
        product_lines_count=("business_line", "nunique")
    )
    .reset_index()
)

# Count purchased policies by individual product
product_counts = (
    policy_products.groupby(
        ["customer_id", "product_id"]
    )["policy_id"]
    .nunique()
    .unstack(fill_value=0)
)

# Ensure all six products are represented as columns
product_ids = [
    "TRV001", "TRV002",
    "PET001", "PET002",
    "HOME001", "HOME002"
]

product_counts = product_counts.reindex(
    columns=product_ids,
    fill_value=0
)

product_counts = product_counts.rename(columns={
    "TRV001": "policies_travel_single_trip",
    "TRV002": "policies_travel_annual",
    "PET001": "policies_pet_standard",
    "PET002": "policies_pet_premium",
    "HOME001": "policies_home_standard",
    "HOME002": "policies_home_premium"
}).reset_index()

# Build customer-level product profile
product_profile = (
    customers[["customer_id"]]
    .merge(
        product_summary,
        on="customer_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        product_counts,
        on="customer_id",
        how="left",
        validate="one_to_one"
    )
)

# Handle customers without purchases
product_columns = [
    "policies_travel_single_trip",
    "policies_travel_annual",
    "policies_pet_standard",
    "policies_pet_premium",
    "policies_home_standard",
    "policies_home_premium"
]

product_profile[
    ["product_lines_count"] + product_columns
] = (
    product_profile[
        ["product_lines_count"] + product_columns
    ]
    .fillna(0)
    .astype(int)
)

# Assign mutually exclusive product profiles
product_profile["product_profile"] = np.select(
    [
        product_profile["product_lines_count"] == 0,
        product_profile["product_lines_count"] == 1,
        product_profile["product_lines_count"] >= 2
    ],
    [
        "No Purchase",
        "Single Product",
        "Multi Product"
    ],
    default="Unclassified"
)

# Validate customer coverage and purchased policy counts
assert product_profile["customer_id"].is_unique
assert len(product_profile) == len(customers)
assert (product_profile["product_profile"] != "Unclassified").all()

assert (
    product_profile[product_columns].sum().sum()
    == policies["policy_id"].nunique()
)



### Product Profile Distribution

The following summary shows how the complete registered customer population is distributed across purchasing and product-diversification profiles.

The distribution distinguishes customers without purchases from those purchasing within one or multiple insurance business lines.


In [19]:

# Product profile distribution across all registered customers
product_order = ["No Purchase", "Single Product", "Multi Product"]

product_distribution = (
    product_profile["product_profile"]
    .value_counts()
    .reindex(product_order, fill_value=0)
    .rename_axis("Product Profile")
    .reset_index(name="Customers")
)

product_distribution["Customer Share"] = (
    product_distribution["Customers"] / len(product_profile)
)

display(
    product_distribution.style.format({
        "Customers": "{:,.0f}",
        "Customer Share": "{:.1%}"
    })
)


,Product Profile,Customers,Customer Share
0,No Purchase,"433,006",68.6%
1,Single Product,"192,411",30.5%
2,Multi Product,"5,662",0.9%



### Product Profiling — Key Findings

The Product Profiling analysis identifies three mutually exclusive customer groups across the complete registered population.

- **No Purchase:** 433,006 customers (68.6%) have no recorded policy purchases.
- **Single Product:** 192,411 customers (30.5%) have purchased policies within one business line.
- **Multi Product:** 5,662 customers (0.9%) have purchased policies across two or more business lines.

Among purchasing customers, approximately **97.1% are single-product customers**, while only **2.9% have purchased across multiple business lines**.

This highlights limited observed product diversification and provides a starting point for investigating potential cross-selling opportunities.

However, product diversification should be interpreted in the context of customer needs, product eligibility, and purchasing behavior. A single-product relationship does not necessarily indicate an unrealized commercial opportunity.



## Channel Profiling

### Methodological Approach

Channel Profiling describes how customers purchase insurance policies across Wossist's sales channels.

The analysis distinguishes between:

- **Acquisition Channel:** the channel associated with the customer's initial acquisition, recorded in the customer master data.
- **Sales Channel:** the channel through which an insurance policy was purchased, recorded in policy transactions.

These dimensions are kept separate because customer acquisition and policy purchasing do not necessarily occur through the same channel.

Wossist operates through four sales channels: **Web, Agency, Partner, and Assisted Sales**.

Customers are classified into three mutually exclusive profiles:

| Channel Profile | Definition |
|---|---|
| **No Purchase** | No purchased policies and therefore no observed purchasing channel |
| **Single Channel** | Policies purchased through one distinct sales channel |
| **Multi Channel** | Policies purchased through two or more distinct sales channels |

Channel diversification is measured through the number of distinct purchasing channels, rather than the number of policies.

### Sales Channel Attributes

In addition to the overall Channel Profile, the dataset retains the number of purchased policies associated with each of Wossist's four sales channels.

These attributes are stored as separate numerical columns to support customer filtering and channel-level analysis in the final Power BI report.

The acquisition channel is retained as a separate customer attribute.

The resulting profiles describe observed purchasing behavior rather than the complete customer journey. Purchasing through multiple sales channels does not necessarily imply an omnichannel experience.


In [20]:

# Associate purchased policies with sales channel information
policy_channels = policies[
    ["customer_id", "policy_id", "sales_channel_id"]
].merge(
    sales_channels[
        ["sales_channel_id", "sales_channel_name"]
    ],
    on="sales_channel_id",
    how="left",
    validate="many_to_one"
)

# Validate sales channel mapping
assert policy_channels["sales_channel_name"].notna().all()

# Count distinct purchasing channels per customer
channel_summary = (
    policy_channels.groupby("customer_id")
    .agg(
        sales_channels_count=("sales_channel_id", "nunique")
    )
    .reset_index()
)

# Count purchased policies by individual sales channel
channel_counts = (
    policy_channels.groupby(
        ["customer_id", "sales_channel_id"]
    )["policy_id"]
    .nunique()
    .unstack(fill_value=0)
)

# Ensure all four sales channels are represented
channel_ids = ["WEB", "AGENCY", "PARTNER", "ASSISTED"]

channel_counts = channel_counts.reindex(
    columns=channel_ids,
    fill_value=0
)

channel_counts = channel_counts.rename(columns={
    "WEB": "policies_web",
    "AGENCY": "policies_agency",
    "PARTNER": "policies_partner",
    "ASSISTED": "policies_assisted"
}).reset_index()

# Build customer-level channel profile
channel_profile = (
    customers[
        ["customer_id", "acquisition_channel"]
    ]
    .merge(
        channel_summary,
        on="customer_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        channel_counts,
        on="customer_id",
        how="left",
        validate="one_to_one"
    )
)

# Handle customers without purchases
channel_columns = [
    "policies_web",
    "policies_agency",
    "policies_partner",
    "policies_assisted"
]

channel_profile[
    ["sales_channels_count"] + channel_columns
] = (
    channel_profile[
        ["sales_channels_count"] + channel_columns
    ]
    .fillna(0)
    .astype(int)
)

# Assign mutually exclusive purchasing channel profiles
channel_profile["channel_profile"] = np.select(
    [
        channel_profile["sales_channels_count"] == 0,
        channel_profile["sales_channels_count"] == 1,
        channel_profile["sales_channels_count"] >= 2
    ],
    [
        "No Purchase",
        "Single Channel",
        "Multi Channel"
    ],
    default="Unclassified"
)

# Validate customer coverage and purchased policy counts
assert channel_profile["customer_id"].is_unique
assert len(channel_profile) == len(customers)
assert (channel_profile["channel_profile"] != "Unclassified").all()

assert (
    channel_profile[channel_columns].sum().sum()
    == policies["policy_id"].nunique()
)

# Verify purchasing population consistency with Product Profiling
assert (
    channel_profile.set_index("customer_id")["channel_profile"].eq("No Purchase")
    ==
    product_profile.set_index("customer_id")["product_profile"].eq("No Purchase")
).all()



### Channel Profile Distribution

The following summary shows how registered customers are distributed according to the number of distinct sales channels used for policy purchases.

The distribution provides an overview of purchasing-channel diversification and distinguishes customers without purchases from single-channel and multi-channel purchasers.


In [21]:

# Channel profile distribution across all registered customers
channel_order = ["No Purchase", "Single Channel", "Multi Channel"]

channel_distribution = (
    channel_profile["channel_profile"]
    .value_counts()
    .reindex(channel_order, fill_value=0)
    .rename_axis("Channel Profile")
    .reset_index(name="Customers")
)

channel_distribution["Customer Share"] = (
    channel_distribution["Customers"] / len(channel_profile)
)

display(
    channel_distribution.style.format({
        "Customers": "{:,.0f}",
        "Customer Share": "{:.1%}"
    })
)


,Channel Profile,Customers,Customer Share
0,No Purchase,"433,006",68.6%
1,Single Channel,"188,007",29.8%
2,Multi Channel,"10,066",1.6%



### Channel Profiling — Key Findings

The Channel Profiling analysis identifies three mutually exclusive purchasing-channel profiles across the complete registered customer population.

- **No Purchase:** 433,006 customers (68.6%) have no observed purchasing channel.
- **Single Channel:** 188,007 customers (29.8%) have purchased policies through one sales channel.
- **Multi Channel:** 10,066 customers (1.6%) have purchased policies through two or more distinct sales channels.

Among purchasing customers, approximately **94.9% are single-channel customers**, while **5.1% have purchased through multiple sales channels**.

Multi-channel purchasing is more common than multi-product purchasing (5.1% versus 2.9% of purchasing customers), although both remain relatively uncommon.

These results describe observed purchasing-channel diversification rather than the complete customer journey. Customers may interact with Wossist through multiple touchpoints without necessarily purchasing through multiple channels.

The distinction between acquisition and sales channels will support further analysis of customer acquisition and purchasing behavior in Power BI.



## Lifecycle Profiling

### Methodological Approach

Lifecycle Profiling describes each customer's observed progression through Wossist's commercial journey, from registration to repeat purchasing.

Unlike RFM segmentation, which applies only to purchasing customers, Lifecycle Profiling covers the **entire registered customer population**.

The analysis combines customer registrations, quotation activity, and purchased policies to identify the most advanced commercial stage observed for each customer.

Four mutually exclusive lifecycle profiles are defined:

| Lifecycle Profile | Definition |
|---|---|
| **Registered** | Registered customers with no recorded quotes or purchases |
| **Quoted** | Customers with at least one recorded quote but no purchases |
| **First Purchase** | Customers with exactly one purchased policy |
| **Repeat Purchase** | Customers with two or more purchased policies |

Lifecycle stages are assigned hierarchically, with purchasing activity taking precedence over quotation activity.

A customer who has purchased a policy is classified according to their number of purchases, regardless of previous quotation activity.

**Interpretation notes:**

- *First Purchase* refers to one observed policy purchase, not necessarily a recently acquired customer.
- *Repeat Purchase* identifies multiple policy purchases within the observation period and does not necessarily imply policy renewal or long-term loyalty.
- *Quoted* customers have demonstrated quotation activity but have no recorded purchases.
- Lifecycle profiles represent observed commercial activity rather than predictions of future conversion or churn.

The resulting classification provides a customer-level perspective on commercial progression and can support further conversion and customer development analysis in Power BI.


In [22]:

# Aggregate quotation activity at customer level
quote_summary = (
    quotes.groupby("customer_id")
    .agg(
        quotes_count=("quote_id", "nunique")
    )
    .reset_index()
)

# Aggregate purchasing activity at customer level
purchase_summary = (
    policies.groupby("customer_id")
    .agg(
        policies_count=("policy_id", "nunique")
    )
    .reset_index()
)

# Preserve the complete registered customer population
lifecycle_profile = (
    customers[["customer_id"]]
    .merge(
        quote_summary,
        on="customer_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        purchase_summary,
        on="customer_id",
        how="left",
        validate="one_to_one"
    )
)

# Replace missing activity counts with zero
lifecycle_profile[["quotes_count", "policies_count"]] = (
    lifecycle_profile[["quotes_count", "policies_count"]]
    .fillna(0)
    .astype(int)
)

# Assign the most advanced observed lifecycle stage
lifecycle_profile["lifecycle_profile"] = np.select(
    [
        lifecycle_profile["policies_count"] >= 2,
        lifecycle_profile["policies_count"] == 1,
        lifecycle_profile["quotes_count"] >= 1
    ],
    [
        "Repeat Purchase",
        "First Purchase",
        "Quoted"
    ],
    default="Registered"
)

# Validate customer coverage and purchasing consistency
assert lifecycle_profile["customer_id"].is_unique
assert len(lifecycle_profile) == len(customers)

assert (
    lifecycle_profile["policies_count"].gt(0).sum()
    == len(rfm)
)

assert (
    lifecycle_profile["policies_count"].sum()
    == len(policies)
)



### Lifecycle Profile Distribution

The following summary describes the distribution of registered customers across the four observed commercial lifecycle stages.

The classification distinguishes customers who have not progressed beyond registration, customers who have requested quotations without purchasing, and customers with one or multiple policy purchases.

The results provide an overview of the customer base's commercial progression during the observation period.


In [23]:

# Lifecycle distribution across all registered customers
lifecycle_order = [
    "Registered",
    "Quoted",
    "First Purchase",
    "Repeat Purchase"
]

lifecycle_distribution = (
    lifecycle_profile["lifecycle_profile"]
    .value_counts()
    .reindex(lifecycle_order, fill_value=0)
    .rename_axis("Lifecycle Profile")
    .reset_index(name="Customers")
)

lifecycle_distribution["Customer Share"] = (
    lifecycle_distribution["Customers"] / len(lifecycle_profile)
)

display(
    lifecycle_distribution.style.format({
        "Customers": "{:,.0f}",
        "Customer Share": "{:.1%}"
    })
)


,Lifecycle Profile,Customers,Customer Share
0,Registered,"257,185",40.8%
1,Quoted,"175,821",27.9%
2,First Purchase,"182,966",29.0%
3,Repeat Purchase,"15,107",2.4%



### Lifecycle Profiling — Key Findings

The Lifecycle Profiling analysis identifies four mutually exclusive commercial stages across the complete registered customer population.

- **Registered:** 257,185 customers (40.8%) have no recorded quotations or policy purchases.
- **Quoted:** 175,821 customers (27.9%) have requested at least one quotation without completing a recorded purchase.
- **First Purchase:** 182,966 customers (29.0%) have purchased exactly one policy.
- **Repeat Purchase:** 15,107 customers (2.4%) have purchased two or more policies.

Overall, **68.6% of registered customers have not purchased a policy**, while **31.4% have completed at least one purchase**.

Among purchasing customers, approximately **92.4% have purchased a single policy**, while **7.6% have made multiple purchases**.

The Quoted segment represents a substantial population with observed pre-purchase activity but no recorded conversion, providing a relevant area for further investigation in Power BI.

However, these lifecycle stages describe the most advanced commercial activity observed within the dataset. They do not measure conversion rates, establish customer intent, or predict future purchasing behavior.

Similarly, repeat purchasing should not automatically be interpreted as long-term loyalty, particularly for insurance products with different purchasing cycles.



## Engagement Profiling

### Methodological Approach

Engagement Profiling describes customers' recorded interactions with Wossist across Email, Phone, and Web channels.

Unlike Product, Channel, and Lifecycle Profiling, this analysis does not assign customers to mutually exclusive segments. Instead, it provides a set of independent customer-level indicators that can be combined flexibly in the final Power BI report.

The analysis focuses on three complementary dimensions:

- **Interaction Volume:** the number of recorded events by channel and interaction type.
- **Customer Responsiveness:** observed email engagement and telephone interest, measured through event-based response rates.
- **Interaction Recency:** the number of days since the latest recorded event of each relevant type.

### Channel-Specific Indicators

**Email**

Email activity is measured through sent, opened, and clicked events.

Two response rates are calculated:

- **Email Open Rate:** Email Opened / Email Sent
- **Email Click Rate:** Email Clicked / Email Sent

These are event-based ratios rather than message-level delivery or engagement rates, since individual email message identifiers are not available.

**Phone**

Telephone activity is classified using three recorded outcomes:

- Interested
- No Response
- Not Interested

**Phone Interest Rate** is calculated as Interested / Total Phone Calls, including unanswered calls in the denominator.

**Web**

Web engagement is represented by recorded advertising clicks.

A click-through rate is not calculated because advertising impressions are not available.

### Recency and Missing Values

Recency indicators are calculated relative to **December 31, 2026**, consistently with the RFM analysis.

Customers without a recorded event of a given type receive:

- A count of **0**.
- A missing rate (`NaN`) when no relevant denominator exists.
- A missing recency value (`NaN`) when the event has never been observed.

No SMS indicators are included because the available interaction dataset contains no SMS events.

The resulting indicators describe observed interactions and responsiveness, without implying customer preferences, marketing consent, or future purchasing intentions.


In [26]:

# Reference date aligned with the RFM analysis
REFERENCE_DATE = pd.Timestamp("2026-12-31")

# Prepare interaction dates
interaction_data = interactions.copy()

interaction_data["interaction_date"] = pd.to_datetime(
    interaction_data["interaction_date"]
)

# Define event types used in Engagement Profiling
event_definitions = {
    "email_sent": ("Email", "Email Sent", "Delivered"),
    "email_open": ("Email", "Email Opened", "Opened"),
    "email_click": ("Email", "Email Clicked", "Positive"),
    "phone_interested": ("Phone", "Phone Call", "Interested"),
    "phone_no_response": ("Phone", "Phone Call", "No Response"),
    "phone_not_interested": ("Phone", "Phone Call", "Not Interested"),
    "web_ad_click": ("Web", "Ad Click", "Positive")
}

# Aggregate interaction counts and latest event dates
event_summaries = []

for event_name, (channel, event_type, outcome) in event_definitions.items():

    event_data = interaction_data.loc[
        (interaction_data["interaction_channel"] == channel)
        & (interaction_data["interaction_type"] == event_type)
        & (interaction_data["outcome"] == outcome)
    ]

    summary = (
        event_data.groupby("customer_id")
        .agg(
            **{
                f"{event_name}_count": (
                    "interaction_id", "nunique"
                ),
                f"last_{event_name}_date": (
                    "interaction_date", "max"
                )
            }
        )
        .reset_index()
    )

    event_summaries.append(summary)

# Build one customer-level record
engagement_profile = customers[["customer_id"]].copy()

for summary in event_summaries:
    engagement_profile = engagement_profile.merge(
        summary,
        on="customer_id",
        how="left",
        validate="one_to_one"
    )

# Rename count columns to final reporting names
engagement_profile = engagement_profile.rename(columns={
    "email_sent_count": "email_sent_count",
    "email_open_count": "email_open_count",
    "email_click_count": "email_click_count",
    "phone_interested_count": "phone_interested_count",
    "phone_no_response_count": "phone_no_response_count",
    "phone_not_interested_count": "phone_not_interested_count",
    "web_ad_click_count": "web_ad_click_count"
})

# Replace missing event counts with zero
count_columns = [
    "email_sent_count",
    "email_open_count",
    "email_click_count",
    "phone_interested_count",
    "phone_no_response_count",
    "phone_not_interested_count",
    "web_ad_click_count"
]

engagement_profile[count_columns] = (
    engagement_profile[count_columns]
    .fillna(0)
    .astype(int)
)

# Calculate total phone calls
engagement_profile["phone_call_count"] = (
    engagement_profile["phone_interested_count"]
    + engagement_profile["phone_no_response_count"]
    + engagement_profile["phone_not_interested_count"]
)

# Calculate event-based response rates
engagement_profile["email_open_rate"] = (
    engagement_profile["email_open_count"]
    / engagement_profile["email_sent_count"].replace(0, np.nan)
)

engagement_profile["email_click_rate"] = (
    engagement_profile["email_click_count"]
    / engagement_profile["email_sent_count"].replace(0, np.nan)
)

engagement_profile["phone_interest_rate"] = (
    engagement_profile["phone_interested_count"]
    / engagement_profile["phone_call_count"].replace(0, np.nan)
)

# Calculate recency indicators
recency_mapping = {
    "last_email_sent_date": "days_since_last_email_sent",
    "last_email_open_date": "days_since_last_email_open",
    "last_email_click_date": "days_since_last_email_click",
    "last_phone_interested_date": "days_since_last_phone_interested",
    "last_web_ad_click_date": "days_since_last_web_ad_click"
}

for date_column, recency_column in recency_mapping.items():
    engagement_profile[recency_column] = (
        REFERENCE_DATE - engagement_profile[date_column]
    ).dt.days

# Last phone call regardless of outcome
phone_last_dates = [
    "last_phone_interested_date",
    "last_phone_no_response_date",
    "last_phone_not_interested_date"
]

engagement_profile["last_phone_call_date"] = (
    engagement_profile[phone_last_dates].max(axis=1)
)

engagement_profile["days_since_last_phone_call"] = (
    REFERENCE_DATE - engagement_profile["last_phone_call_date"]
).dt.days

# Keep only the agreed customer-level indicators
engagement_columns = [
    "customer_id",
    "email_sent_count",
    "email_open_count",
    "email_click_count",
    "email_open_rate",
    "email_click_rate",
    "phone_call_count",
    "phone_interested_count",
    "phone_no_response_count",
    "phone_not_interested_count",
    "phone_interest_rate",
    "web_ad_click_count",
    "days_since_last_email_sent",
    "days_since_last_email_open",
    "days_since_last_email_click",
    "days_since_last_phone_call",
    "days_since_last_phone_interested",
    "days_since_last_web_ad_click"
]

engagement_profile = engagement_profile[engagement_columns]

# Validate customer coverage and interaction reconciliation
assert engagement_profile["customer_id"].is_unique
assert len(engagement_profile) == len(customers)

assert (
    engagement_profile["email_sent_count"].sum() == 167698
)
assert (
    engagement_profile["email_open_count"].sum() == 78888
)
assert (
    engagement_profile["email_click_count"].sum() == 37089
)
assert (
    engagement_profile["phone_call_count"].sum() == 75639
)
assert (
    engagement_profile["web_ad_click_count"].sum() == 281116
)

assert (
    engagement_profile[
        ["email_open_rate", "email_click_rate", "phone_interest_rate"]
    ].max().le(1).all()
)

assert (
    engagement_profile[
        ["email_open_rate", "email_click_rate", "phone_interest_rate"]
    ].min().ge(0).all()
)



### Engagement Indicators — Summary

The following summary provides an overview of recorded interaction activity and customer responsiveness across Email, Phone, and Web.

Unlike the previous profiling dimensions, Engagement is not represented by a single categorical distribution. Instead, the summary reports customer coverage, interaction volumes, and aggregate response rates.

Customer-level indicators remain available independently for subsequent analysis in Power BI.


In [27]:

# Summarize engagement indicators across the customer base
engagement_summary = pd.DataFrame([
    {
        "Metric": "Customers with Recorded Interactions",
        "Value": f"{interactions['customer_id'].nunique():,}"
    },
    {
        "Metric": "Customers without Recorded Interactions",
        "Value": f"{len(customers) - interactions['customer_id'].nunique():,}"
    },
    {
        "Metric": "Customers with Email Sends",
        "Value": f"{(engagement_profile['email_sent_count'] > 0).sum():,}"
    },
    {
        "Metric": "Email Open Rate (Overall)",
        "Value": (
            f"{engagement_profile['email_open_count'].sum() / engagement_profile['email_sent_count'].sum():.1%}"
        )
    },
    {
        "Metric": "Email Click Rate (Overall)",
        "Value": (
            f"{engagement_profile['email_click_count'].sum() / engagement_profile['email_sent_count'].sum():.1%}"
        )
    },
    {
        "Metric": "Customers with Phone Calls",
        "Value": f"{(engagement_profile['phone_call_count'] > 0).sum():,}"
    },
    {
        "Metric": "Phone Interest Rate (Overall)",
        "Value": (
            f"{engagement_profile['phone_interested_count'].sum() / engagement_profile['phone_call_count'].sum():.1%}"
        )
    },
    {
        "Metric": "Customers with Web Ad Clicks",
        "Value": f"{(engagement_profile['web_ad_click_count'] > 0).sum():,}"
    }
])

display(engagement_summary)


,Metric,Value
0,Customers with Recorded Interactions,"158,663"
1,Customers without Recorded Interactions,"472,416"
2,Customers with Email Sends,"99,746"
3,Email Open Rate (Overall),47.0%
4,Email Click Rate (Overall),22.1%
5,Customers with Phone Calls,"60,907"
6,Phone Interest Rate (Overall),45.7%
7,Customers with Web Ad Clicks,"122,124"



### Engagement Profiling — Key Findings

Engagement Profiling provides a customer-level view of recorded interactions and responsiveness across Email, Phone, and Web channels.

**Interaction Coverage**

- **158,663 customers (25.1%)** have at least one recorded interaction, while **472,416 customers (74.9%)** have no recorded interactions during the observation period.
- The absence of recorded interactions does not necessarily indicate customer inactivity or lack of interest.

**Email Engagement**

- **99,746 customers** have received at least one recorded email.
- The overall **Email Open Rate is 47.0%**, while the **Email Click Rate is 22.1%**, calculated as recorded openings and clicks divided by email sends.
- These event-based indicators describe observed email responsiveness but should not be interpreted as unique-message engagement rates.

**Phone Engagement**

- **60,907 customers** have at least one recorded phone call.
- The overall **Phone Interest Rate is 45.7%**, representing the proportion of calls with an Interested outcome.
- Phone outcomes distinguish explicit interest, lack of response, and expressed disinterest, enabling more detailed customer-level analysis.

**Web Engagement**

- **122,124 customers** have at least one recorded advertising click.
- Web clicks provide an additional signal of observed digital activity, although advertising impressions are unavailable for calculating click-through rates.

**Business Implications**

The resulting indicators enable customer-level comparisons between purchasing behavior, marketing responsiveness, and interaction recency.

Rather than assigning a single engagement label, the analysis preserves independent metrics that can be combined with RFM, Product, Channel, and Lifecycle Profiling in the final Power BI report.

This approach supports more flexible customer exploration without introducing arbitrary engagement thresholds.



## Customer-Level Integration

### Final Dataset Construction

The four profiling dimensions are consolidated into a single customer-level analytical dataset, enriched with the previously developed RFM segmentation.

The final dataset combines:

- **Customer Attributes:** demographic, geographic, registration, and acquisition information.
- **Product Profiling:** business-line diversification and purchased policy counts across six product variants.
- **Channel Profiling:** purchasing-channel diversification and purchased policy counts across four sales channels.
- **Lifecycle Profiling:** customer progression based on recorded quotations and purchases.
- **Engagement Profiling:** interaction volumes, response rates, and recency indicators across Email, Phone, and Web.
- **RFM Segmentation:** purchasing recency, frequency, monetary value, and the corresponding RFM segment.

The dataset maintains a granularity of **one row per registered customer**.

RFM information is available only for customers with at least one recorded purchase. Customers without purchases retain missing RFM values rather than receiving an artificial RFM classification.

This consolidated structure supports flexible customer filtering, cross-segmentation, and reporting in Power BI.


In [30]:

# Build the final customer-level analytical dataset
customer_business_profiling = (
    customers
    .merge(
        product_profile,
        on="customer_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        channel_profile.drop(columns=["acquisition_channel"]),
        on="customer_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        lifecycle_profile,
        on="customer_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        engagement_profile,
        on="customer_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        rfm,
        on="customer_id",
        how="left",
        validate="one_to_one"
    )
)

# Calculate customer age at the analysis reference date
reference_date = pd.Timestamp("2026-12-31")

birth_dates = pd.to_datetime(
    customer_business_profiling["birth_date"],
    errors="coerce"
)

customer_business_profiling["customer_age"] = (
    reference_date.year
    - birth_dates.dt.year
    - (
        (birth_dates.dt.month > reference_date.month)
        | (
            (birth_dates.dt.month == reference_date.month)
            & (birth_dates.dt.day > reference_date.day)
        )
    ).astype(int)
)

# Remove customer names and birth dates from the analytical export
customer_business_profiling = customer_business_profiling.drop(
    columns=["first_name", "last_name", "birth_date"]
)

# Place customer identifiers and main profiling dimensions first
main_columns = [
    "customer_id",
    "customer_age",
    "gender",
    "country",
    "region",
    "city",
    "registration_date",
    "acquisition_channel",
    "product_profile",
    "channel_profile",
    "lifecycle_profile",
    "rfm_segment"
]

remaining_columns = [
    column
    for column in customer_business_profiling.columns
    if column not in main_columns
]

customer_business_profiling = customer_business_profiling[
    main_columns + remaining_columns
]



### Final Dataset Validation

Before exporting the analytical dataset, the following checks verify:

- Complete coverage of the registered customer population.
- Uniqueness of the customer identifier.
- Consistency between purchasing profiles, lifecycle stages, and RFM coverage.
- Preservation of recorded policy counts across product and sales channel attributes.

These checks ensure that the consolidated dataset maintains its intended customer-level granularity and that the profiling dimensions remain consistent with the underlying transactional data.


In [31]:

# Customer coverage and uniqueness
assert len(customer_business_profiling) == len(customers)
assert customer_business_profiling["customer_id"].is_unique

# RFM coverage must match the purchasing population
purchasing_customers = (
    customer_business_profiling["policies_count"] > 0
)

assert purchasing_customers.sum() == len(rfm)

assert (
    customer_business_profiling["rfm_segment"].notna()
    == purchasing_customers
).all()

# Purchasing profile consistency
assert (
    customer_business_profiling["product_profile"].eq("No Purchase")
    == ~purchasing_customers
).all()

assert (
    customer_business_profiling["channel_profile"].eq("No Purchase")
    == ~purchasing_customers
).all()

assert (
    customer_business_profiling["lifecycle_profile"].isin(
        ["First Purchase", "Repeat Purchase"]
    )
    == purchasing_customers
).all()

# Reconcile policy counts across profiling dimensions
assert (
    customer_business_profiling[product_columns].sum().sum()
    == len(policies)
)

assert (
    customer_business_profiling[channel_columns].sum().sum()
    == len(policies)
)

assert (
    customer_business_profiling["policies_count"].sum()
    == len(policies)
)

# Summarize the final dataset
final_dataset_summary = pd.DataFrame({
    "Metric": [
        "Total Customers",
        "Dataset Columns",
        "Purchasing Customers",
        "Non-Purchasing Customers",
        "Customers with RFM Segmentation",
        "Unique Customer IDs"
    ],
    "Value": [
        len(customer_business_profiling),
        customer_business_profiling.shape[1],
        purchasing_customers.sum(),
        (~purchasing_customers).sum(),
        customer_business_profiling["rfm_segment"].notna().sum(),
        customer_business_profiling["customer_id"].nunique()
    ]
})

display(
    final_dataset_summary.style.format({
        "Value": "{:,.0f}"
    })
)


,Metric,Value
0,Total Customers,"631,079"
1,Dataset Columns,50
2,Purchasing Customers,"198,073"
3,Non-Purchasing Customers,"433,006"
4,Customers with RFM Segmentation,"198,073"
5,Unique Customer IDs,"631,079"



## Final Dataset Export

The validated customer-level dataset is exported as a compressed CSV file for subsequent use in Power BI.

**Output:** `customer_business_profiling.csv.gz`

The export preserves all registered customers and the analytical attributes developed throughout the profiling process.

The compressed file is stored in the project's `output/` directory and subsequently uploaded to the GitHub repository, ensuring that the Power BI reporting layer can reuse the finalized analytical dataset.

The notebook therefore provides a reproducible workflow from source data to customer-level business profiling.


In [33]:

from pathlib import Path

# Create the output directory if necessary
output_dir = Path("output")
output_dir.mkdir(parents=True, exist_ok=True)

# Export the consolidated customer-level dataset
output_path = output_dir / "customer_business_profiling.csv.gz"

customer_business_profiling.to_csv(
    output_path,
    index=False,
    compression="gzip"
)

print(f"Dataset exported successfully: {output_path}")
print(f"Rows: {len(customer_business_profiling):,}")
print(f"Columns: {customer_business_profiling.shape[1]}")


Dataset exported successfully: output\customer_business_profiling.csv.gz
Rows: 631,079
Columns: 50
